# Market risk via EVT + t-copula

_A runnable, offline version of the GARCH/EVT/copula tail-risk pipeline._

**pyportfolios.com** · [/research/evt-t-copula-var](https://pyportfolios.com/research/evt-t-copula-var)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## Note

The article filters volatility with a **GARCH-t** (`arch`). To keep this notebook
dependency-light and runnable anywhere, we use an **EWMA** volatility filter as a
stand-in. Swap in `arch_model(..., vol='GARCH', dist='t')` for production.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import genpareto, t as student_t, rankdata, chi2

rng = np.random.default_rng(2)
T, N, nu_true = 2000, 3, 5
corr_true = np.array([[1, .5, .3], [.5, 1, .4], [.3, .4, 1.0]])
L = np.linalg.cholesky(corr_true)

# correlated Student-t shocks -> fat tails + tail dependence
g = (L @ rng.standard_normal((N, T))).T
mix = np.sqrt(nu_true / chi2.rvs(nu_true, size=(T, 1), random_state=rng))
rets = pd.DataFrame(g * mix * 0.01, columns=["A", "B", "C"])
weights = np.array([0.4, 0.35, 0.25])

## 1) EWMA filter  2) EVT (POT) margins  3) t-copula  4) Monte Carlo VaR

In [ ]:
# 1) volatility filter -> standardised residuals
ewm_var = rets.ewm(span=33).var()
Z = (rets / np.sqrt(ewm_var)).dropna()
sigma_fcast = np.sqrt(ewm_var.iloc[-1].values)

# 2) semi-parametric margins: empirical body + GPD (Pareto) upper tail
def fit_pot(z, q=0.95):
    u = np.quantile(z, q)
    xi, _, beta = genpareto.fit(z[z > u] - u, floc=0)
    return u, xi, beta

tails = {c: fit_pot(Z[c].values) for c in Z}

def inv_margin(u, z, tail):
    out = np.quantile(z, u)
    hi = u > 0.95
    u_t, xi, beta = tail
    out[hi] = u_t + genpareto.ppf((u[hi] - 0.95) / 0.05, xi, scale=beta)
    return out

# 3) t-copula on the pseudo-observations
U = Z.apply(lambda s: rankdata(s) / (len(s) + 1)).values
nu = 6
corr = np.corrcoef(student_t.ppf(U, df=nu), rowvar=False)

# 4) simulate, invert margins, re-inflate by forecast vol
def t_copula_sample(corr, nu, n):
    Lc = np.linalg.cholesky(corr)
    z = (Lc @ rng.standard_normal((corr.shape[0], n))).T
    w = np.sqrt(nu / chi2.rvs(nu, size=(n, 1), random_state=rng))
    return student_t.cdf(z * w, df=nu)

n = 100_000
Us = t_copula_sample(corr, nu, n)
sim = np.column_stack([inv_margin(Us[:, k], Z[c].values, tails[c]) for k, c in enumerate(Z)])
sim_rets = sim * sigma_fcast

pnl = sim_rets @ weights
loss = -pnl
for a in (0.95, 0.99):
    var = np.quantile(loss, a)
    cvar = loss[loss >= var].mean()
    print(f"{a:.0%}   VaR {var:6.2%}   CVaR {cvar:6.2%}")

# contrast with a naive Gaussian VaR
mu, sd = pnl.mean(), pnl.std()
from scipy.stats import norm
print(f"\nGaussian 99% VaR (understates): {-(mu + sd * norm.ppf(0.01)):6.2%}")